# Unsupervised Text Classification

Using LLMs to categorize responses seems to be pretty unreliable and I have been unsuccessful in getting the models to cleanly work. 
In this document, I discuss using `BERTopic` to create categories of responses. This seems to be a fairly popular method in recent years.



## Questions

All responses can be found in `data/student-responses.csv`. In this file, all collected responses are stored from the bar chart and heatmap experiments. In the data cleaning script, a flag for bar chart or heatmap experiment was included.

In [ ]:
# Import modules
import pandas as pd

# Read dataset
df = pd.read_csv("../data/student-responses.csv")

# Filter only Bar Chart experiment responses
df = df[df["experiment"] == "Bar chart"]
df.head()

In [ ]:
df["section"].value_counts().to_frame(name="count").reset_index().rename(columns={"index": "section"}).sort_values("section")

Next, it is worth noting that the quesitons are not in order. Below is a table summary of the questions and their corresponding modules.

| Module | Question Number | Dataframe Column Index (0-index) | Prompt |
|---|---:|---:|---|
| pre-experiment | Q1 | 13 | In this class, you’ll be learning about the process of scientific investigation. What do you think that process looks like, from the perspective of a researcher, compared to what it looks like from the perspective of someone in the general public who is a consumer of scientific results? Write a paragraph (at least 3-5 sentences) about how you think science happens. |
| post-experiment | Q2 | 8 | What do you think the purpose of the experiment was? |
| post-experiment | Q3 | 10 | What hypotheses might the experimenter have been testing? |
| post-experiment | Q4 | 11 | What sources of error are involved in this experiment? |
| post-experiment | Q5 | 12 | What variables were examined? For each variable, identify whether it was quantitative or categorical. |
| post-experiment | Q6 | 9 | What elements of experimental design, such as randomization or the use of a control group, do you think were present in the experiment? Why? |
| abstract reflection | Q7 | 4 | What components of the experiment are clearer now than they were as a participant? What questions do you still have for the experimenter? Write 3-5 sentences reflecting on the abstract. |
| presentation reflection | Q8 | 14 | How did the information you gained from the components of this project (participation, post-study reflection, extended abstract, presentation) differ? |
| presentation reflection | Q9 | 16 | What components were emphasized in the presentation that weren’t emphasized in the abstract? Why do you think that is? |
| presentation reflection | Q10 | 17 | What critiques do you have of this study and its design? What would have made the study better? |
| presentation reflection | Q11 | 15 | If you had to hear about this study using only the extended abstract or only the presentation, which one would you prefer? Which one would be better for determining whether the experiment was well designed? |

## Side note

It looks like some students may have used AI in their answers. While I doubt that we could detect human text (tools seem to be unreliable for this task), we may want to remove these participants if their answers are completely off topic. 

- ac6eb19c384b5ec70af8f6b086758376: this user had some answers talking about cows, which is completely irrelevant to the sources of error question

## BERTopic

BERTopic is an unsupervised text classification method that leverages clustering and dimensional reduction. 

**Webpage:** <https://maartengr.github.io/BERTopic/index.html>

1. 

**Guide:** <https://www.youtube.com/watch?v=v3SePt3fr9g>

In [ ]:
df.columns[13].split()

In [ ]:
# Modules for topic modeling
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from sentence_transformers import SentenceTransformer
import hdbscan
from umap import UMAP
import openai
from bertopic.representation import OpenAI
from bertopic.representation import KeyBERTInspired
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction import text

# Embedding model
embedding_model = SentenceTransformer("all-mpnet-base-v2")

# UMAP parameters tuned to preserve more neighborhood structure
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

# HDBSCAN tuned to reduce outliers
hdb = hdbscan.HDBSCAN(
    min_cluster_size=3,
    min_samples=1,
    allow_single_cluster=True,
    cluster_selection_method="eom",
    prediction_data=True
)

# Combine custom stop words with stop words from CountVectorizer
column_name = df.columns[4]
stop_words = text.ENGLISH_STOP_WORDS.union(column_name.split())
stop_words = [word for word in stop_words]

# Vectorizer model
vectorizer_model = CountVectorizer(
    #ngram_range=(1, 2),
    #stop_words='english',
    stop_words=stop_words,
    min_df=2,
    max_df=0.95
)

# Topic Representation model
ctfidf_model = ClassTfidfTransformer(
    reduce_frequent_words=True,
    bm25_weighting=True
)

# Configure OpenAI for topic representation
client = openai.OpenAI(
    base_url="http://localhost:11434/v1",
    api_key='ollama'
)

representation_model = OpenAI(client, 
                              system_prompt="You are a helpful assistant that summarizes the main themes of a collection of text responses. You will be given a list of representative responses, and you should generate a concise summary that captures the key themes and insights from those responses. Your summary should be clear, informative, and reflect the main ideas expressed in the representative responses.",
                              model='mistral')

# Alternate representation without LLM
representation_model = KeyBERTInspired()

# Fit model for Q1 responses
q1 = df.iloc[:, 4].dropna().astype(str).str.strip()
q1 = q1[q1.ne("")].tolist()

# Fit BERTopic model
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdb,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    nr_topics="auto",
    calculate_probabilities=True,
    verbose=True,
    representation_model=representation_model
)
topics, probs = topic_model.fit_transform(q1)

In [ ]:
topic_model.get_topic_info()

In [ ]:
#topic_model.reduce_topics(docs=q1, nr_topics=5)
topic_model.update_topics(q1, vectorizer_model=CountVectorizer(ngram_range=(1, 2), stop_words="english"))
topic_model.get_topic_info()

In [ ]:
# Perform hierarchical topic reduction
hierarchical_topics = topic_model.hierarchical_topics(q1)

# Visualize the subtopics
topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

In [ ]:
topic_model.visualize_topics()

In [ ]:
topic_model.visualize_documents(q1, hide_annotations=True)

In [ ]:
%pip install wordcloud
from wordcloud import WordCloud
import matplotlib.pyplot as plt

In [ ]:
def create_wordcloud(model, topic):
    text = {word: value for word, value in model.get_topic(topic)}
    wc = WordCloud(background_color="white", max_words=1000)
    wc.generate_from_frequencies(text)
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.show()

# Show wordcloud
create_wordcloud(topic_model, topic=3)

## Function that fits model for specified question

In [ ]:
# Modules for topic modeling
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from sentence_transformers import SentenceTransformer
import hdbscan
from umap import UMAP
import openai
from bertopic.representation import OpenAI
from bertopic.representation import KeyBERTInspired
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction import text

# Function to fit BERTopic on a specific dataframe column
def fit_bertopic(df, column_index, remove_prompt_words=False):
    """
    Fit BERTopic on one dataframe column.

    Returns:
        topic_model: fitted BERTopic model
        topics: topic assignment list for each document
        probs: topic probability matrix from BERTopic
        docs: list of documents used for fitting
    """
    # Pull selected column, keeping only non-empty responses
    prompt_label = str(df.columns[column_index])
    responses = df.iloc[:, column_index]
    valid_mask = responses.notna() & responses.astype(str).str.strip().ne("")
    docs = responses[valid_mask].astype(str).tolist()

    if len(docs) == 0:
        raise ValueError(f"No non-empty responses found in column index {column_index}.")

    print(f"Running BERTopic for prompt: {prompt_label}")

    # Embedding model setup
    embedding_model = SentenceTransformer("all-mpnet-base-v2")

    # UMAP setup
    umap_model = UMAP(
        n_neighbors=15,
        n_components=5,
        min_dist=0.0,
        metric="cosine",
        random_state=42
    )

    # HDBSCAN setup
    hdb = hdbscan.HDBSCAN(
        min_cluster_size=3,
        min_samples=1,
        allow_single_cluster=True,
        cluster_selection_method="eom",
        prediction_data=True
    )

    # Vectorizer model setup
    # Combine custom stop words with stop words from CountVectorizer
    column_name = df.columns[4]
    stop_words = text.ENGLISH_STOP_WORDS.union(column_name.split())
    stop_words = [word for word in stop_words]

    vectorizer_model = CountVectorizer(
        stop_words=stop_words if remove_prompt_words else "english",
        min_df=2,
        max_df=0.95
    )

    # OpenAI/Ollama-backed representation model setup
    client = openai.OpenAI(
        base_url="http://localhost:11434/v1",
        api_key="ollama"
    )
    representation_model = OpenAI(client, model='mistral') #use LLM to generate topic representations
    #representation_model = KeyBERTInspired() #remove stop words from representation model

    # Fit BERTopic
    topic_model = BERTopic(
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=hdb,
        vectorizer_model=vectorizer_model,
        ctfidf_model=ctfidf_model,
        nr_topics="auto",
        calculate_probabilities=True,
        verbose=False,
        representation_model=representation_model
    )
    topics, probs = topic_model.fit_transform(docs)
    topic_model.update_topics(docs, vectorizer_model=CountVectorizer(ngram_range=(1, 2))) #include bigrams in topic representations

    return topic_model, topics, probs, docs

In [147]:
# Example for Q1 responses (column index 13)
q1_topic_model, q1_topics, q1_probs, q1_docs = fit_bertopic(
    df=df,
    column_index=13
)

Running BERTopic for prompt: In this class you ll be learning about the process of scientific investigation What do you think that process looks like from the perspective of a researcher compared to what it looks like from the perspective of someone in the general public who is a consumer of scientific results Write a paragraph at least 3 5 sentences about how you think science happens


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4931.73it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-04-12 14:38:48,237 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 20/20 [00:07<00:00,  2.63it/s]
2026-04-12 14:38:55,860 - BERTopic - Embedding - Completed ✓
2026-04-12 14:38:55,861 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-12 14:38:56,940 - BERTopic - Dimensionality - Completed ✓
2026-04-12 14:38:56,940 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-12 14:38:57,092 - BERTopic - Cluster - Completed ✓
2026-04-12 14:38:57,093 - BERTopic - Representation - Extracting topics using c-TF-IDF for top

In [149]:
q1_topic_model.visualize_topics()

In [144]:
q1_hierarchical_topics = q1_topic_model.hierarchical_topics(df.iloc[:, 13].dropna().astype(str).tolist())
q1_topic_model.visualize_hierarchy(hierarchical_topics=q1_hierarchical_topics)

NameError: name 'q1_topic_model' is not defined

# Paper Draft

## Methods

**NOTE:** This section is only for these models. I have more in the actual manuscript.

Our sample is composed of students enrolled in STAT 218 at the University of Nebraska--Lincoln. While all students were required to participant in the experiential learning project as part of the course cirriculum, data was collected if students meet the age of majority in Nebraska (age 19 or older) and if they consented to data collection. The data collection took place between Summer 2023 and Spring 2025, where XX sections of STAT 218 participated. 


### Text Classification

In recent years, there has been a growing research area in Large-Language Models (LLMs) for classifying open-ended survey responses. Despite the promising aspect of using LLMs for classification, these models are sensitive to prompt and token limits (XXX), which can greatly influence the outputs. Additionally, these models tend to suffer from hallucinations and/or low accuracy rates compared to traditional human codings (XXX). The current capacity of LLMs are not yet ready for standalone classification, but they do have some integrations with non-zero box methods. 

For the classiciation of our responses, we focus on BERTopic (XXX), which is a topic modeling algorithm that incorporates clustering and dimensional reduction. BERTopic is highly customizable in each stage of the algorithm, allowing for multiple specifications for fine-tuning. Descriptions of this process can be found in Table (XXX), along with our specified models and hyper-parameters for each stage. We note a few of our chosen hyperparameter selections. For the clustering stage with HBDSCAN, we set the minimum cluster size to 5 so that smaller clusters can form. We found that the default settings were too restrictive and formed topics that closely aligned with the original prompt. We also included n-grams up to five words to allow for common phrases that students may have responded with (e.g., "scientific process"). Lastly, we incorporated Mistral (XXX) as a locally-run LLM to fine-tune the generated topic lists into interpretable categories, while also respecting concerns over data privacy of cloud-based LLMs.




| Stage | Step | Purpose | Option Used |
|---|---|---|---|
| Stage 1 | extract embeddings | Convert each response into a numeric vector that captures semantic meaning. | `embedding_model="all-MiniLM-L6-v2"` |
| Stage 2 | reduce dimensionality | Compress embeddings into a lower-dimensional space for better clustering efficiency | `UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=42)` |
| Stage 3 | cluster reduced embeddings | Group similar responses into topic clusters. | `HDBSCAN(min_cluster_size=5, min_samples=2, cluster_selection_epsilon=0.1, prediction_data=True)` |
| Stage 4 | tokenize topics | Break text into candidate terms/phrases used to represent each cluster. | `n_gram_range=(1, 5)` |
| Stage 5 | extract topic words | Compute the most representative words/phrases for each cluster. | BERTopic default c-TF-IDF weighting |
| Stage 6 | fine-tune topic representations | Improve readability and specificity of topic labels/keywords. | `representation_model=OpenAI(client, model="mistral")` with Ollama endpoint `http://localhost:11434/v1` |

## Results

### Pre-Experiment

The pre-experiment prompt consisted of asking participants to write a paragraph about how science differs from researchers and the general public. XXX students provided a response to this prompt.

> In this class, you’ll be learning about the process of scientific investigation. What do you think that process looks like, from the perspective of a researcher, compared to what it looks like from the perspective of someone in the general public who is a consumer of scientific results? Write a paragraph (at least 3-5 sentences) about how you think science happens.


In [ ]:
# Q1
q1_topic_model, q1_topics, q1_probs, q1_docs = fit_bertopic(
    df=df,
    column_index=13,
    remove_prompt_words=True
)

In [ ]:
q1_topic_model.visualize_topics()

In [ ]:
q1_hierarchical_topics = q1_topic_model.hierarchical_topics(q1_docs)
q1_topic_model.visualize_hierarchy(hierarchical_topics=q1_hierarchical_topics)

### Post-Experiment

#### Q2

**Prompt:**
> What do you think the purpose of the experiment was?

In [ ]:
# Q2
q2_topic_model, q2_topics, q2_probs, q2_docs = fit_bertopic(
    df=df,
    column_index=8,
    remove_prompt_words=True
)

In [ ]:
q2_topic_model.visualize_topics()

In [ ]:
q2_hierarchical_topics = q2_topic_model.hierarchical_topics(q2_docs)
q2_topic_model.visualize_hierarchy(hierarchical_topics=q2_hierarchical_topics)

#### Q3

**Prompt:**
> What hypotheses might the experimenter have been testing?

In [ ]:
# Q3
q3_topic_model, q3_topics, q3_probs, q3_docs = fit_bertopic(
    df=df,
    column_index=10,
    remove_prompt_words=True
)

In [ ]:
q3_topic_model.visualize_topics()

In [ ]:
q3_hierarchical_topics = q3_topic_model.hierarchical_topics(q3_docs)
q3_topic_model.visualize_hierarchy(hierarchical_topics=q3_hierarchical_topics)

#### Q4

**Prompt:**
> What sources of error are involved in this experiment?

In [ ]:
# Q4
q4_topic_model, q4_topics, q4_probs, q4_docs = fit_bertopic(
    df=df,
    column_index=11,
    remove_prompt_words=True
)

In [ ]:
q4_topic_model.visualize_topics()

In [ ]:
q4_hierarchical_topics = q4_topic_model.hierarchical_topics(q4_docs)
q4_topic_model.visualize_hierarchy(hierarchical_topics=q4_hierarchical_topics)

#### Q5

**Prompt:**
> What variables were examined? For each variable, identify whether it was quantitative or categorical.

In [ ]:
# Q5
q5_topic_model, q5_topics, q5_probs, q5_docs = fit_bertopic(
    df=df,
    column_index=12,
    remove_prompt_words=False
)

In [ ]:
q5_topic_model.visualize_topics()

In [ ]:
q5_hierarchical_topics = q5_topic_model.hierarchical_topics(q5_docs)
q5_topic_model.visualize_hierarchy(hierarchical_topics=q5_hierarchical_topics)

In [ ]:
q5_topic_model.visualize_document_datamap(q5_docs)

#### Q6

**Prompt:**
> What elements of experimental design, such as randomization or the use of a control group, do you think were present in the experiment? Why?

In [ ]:
# Q6
q6_topic_model, q6_topics, q6_probs, q6_docs = fit_bertopic(
    df=df,
    column_index=9,
    remove_prompt_words=False
)

In [ ]:
q6_topic_model.visualize_topics()

In [ ]:
q6_hierarchical_topics = q6_topic_model.hierarchical_topics(q6_docs)
q6_topic_model.visualize_hierarchy(hierarchical_topics=q6_hierarchical_topics)

### Abstract Reflection

#### Q7

**Prompt:**
> What components of the experiment are clearer now than they were as a participant? What questions do you still have for the experimenter? Write 3-5 sentences reflecting on the abstract.

In [ ]:
# Q7
q7_topic_model, q7_topics, q7_probs, q7_docs = fit_bertopic(
    df=df,
    column_index=4,
    remove_prompt_words=False
)

In [ ]:
q7_topic_model.visualize_topics()

In [ ]:
q7_hierarchical_topics = q7_topic_model.hierarchical_topics(q7_docs)
q7_topic_model.visualize_hierarchy(hierarchical_topics=q7_hierarchical_topics)

### Presentation Reflection

#### Q8

**Prompt:**
> How did the information you gained from the components of this project (participation, post-study reflection, extended abstract, presentation) differ?

In [ ]:
# Q8
q8_topic_model, q8_topics, q8_probs, q8_docs = fit_bertopic(
    df=df,
    column_index=14,
    remove_prompt_words=False
)

In [ ]:
q8_topic_model.visualize_topics()

In [ ]:
q8_hierarchical_topics = q8_topic_model.hierarchical_topics(q8_docs)
q8_topic_model.visualize_hierarchy(hierarchical_topics=q8_hierarchical_topics)

#### Q9

**Prompt:**
> What components were emphasized in the presentation that weren’t emphasized in the abstract? Why do you think that is?

In [ ]:
# Q9
q9_topic_model, q9_topics, q9_probs, q9_docs = fit_bertopic(
    df=df,
    column_index=16,
    remove_prompt_words=True
)

In [ ]:
q9_topic_model.visualize_topics()

In [ ]:
q9_hierarchical_topics = q9_topic_model.hierarchical_topics(q9_docs)
q9_topic_model.visualize_hierarchy(hierarchical_topics=q9_hierarchical_topics)

#### Q10

**Prompt:**
> What critiques do you have of this study and its design? What would have made the study better?

In [ ]:
# Q10
q10_topic_model, q10_topics, q10_probs, q10_docs = fit_bertopic(
    df=df,
    column_index=17,
    remove_prompt_words=True
)

In [ ]:
q10_topic_model.visualize_topics()

In [ ]:
q10_hierarchical_topics = q10_topic_model.hierarchical_topics(q10_docs)
q10_topic_model.visualize_hierarchy(hierarchical_topics=q10_hierarchical_topics)

#### Q11

**Prompt:**
> If you had to hear about this study using only the extended abstract or only the presentation, which one would you prefer? Which one would be better for determining whether the experiment was well designed?

In [ ]:
# Q11
q11_topic_model, q11_topics, q11_probs, q11_docs = fit_bertopic(
    df=df,
    column_index=15,
    remove_prompt_words=False
)

In [ ]:
q11_topic_model.visualize_topics()

In [ ]:
q11_hierarchical_topics = q11_topic_model.hierarchical_topics(q11_docs)
q11_topic_model.visualize_hierarchy(hierarchical_topics=q11_hierarchical_topics)

## Export results